# einops.repeat — procedural drill

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-repeat`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five `einops.repeat` patterns that ramp from new-axis broadcast → per-token-to-per-feature → vertical stretch → horizontal tile → 2×2 nearest-neighbor upsample. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F
import einops
from einops import repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-repeat`**, which bridges to the bank subtopic `Einops: Repeat` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat"
DD_SUBTOPIC = "Einops: Repeat"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## einops.repeat — quick refresher

`repeat(tensor, pattern, **axes_lengths)` introduces new axes or stretches existing ones:
1. **New axis** — `'c h w -> b c h w'` with `b=4` broadcasts across a new batch dim.
2. **Trailing axis** — `'b t -> b t d'` with `d=64` materializes a per-token weight at per-feature width.
3. **Stretch (nearest-neighbor)** — `'h w -> (h r) w'` with `r=2` makes each row appear twice in a block.
4. **Tile** — `'h w -> h (r w)'` with `r=2` concatenates two copies of every row.

Difference between **stretch** `(h r)` and **tile** `(r h)`: the factor written first varies slower. `(h r)` puts source row 0 at positions `0..r-1`; `(r h)` puts source row 0 at positions `0, h, 2h, ...`.

### Exercise 1 — broadcast across batch

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall the `repeat(x, pattern, **size)` call shape for introducing a new named output axis.
> Keywords: add-axis, broadcast, kwarg-binding
> ```

**KCs targeted:** `repeat-add-axis`

Implement `ex1_broadcast_batch(x, b)` to materialize a single `(c, h, w)` image into a batch of `b` identical copies.

Input shape: `(c, h, w)`. Output shape: `(b, c, h, w)`.

Use `einops.repeat` with the new axis bound by kwarg — not `x.unsqueeze(0).expand(...)` or `torch.stack`. The point is to write the pattern.

In [ ]:
def ex1_broadcast_batch(x: Tensor, b: int) -> Tensor:
    """Repeat `x` of shape (c, h, w) into shape (b, c, h, w)."""
    raise NotImplementedError()


def _test_ex1():
    x = t.arange(3 * 4 * 5).reshape(3, 4, 5).float()
    y = ex1_broadcast_batch(x, b=2)
    assert y.shape == (2, 3, 4, 5), f'expected (2,3,4,5), got {y.shape}'
    assert t.equal(y[0], x), 'batch 0 does not match x'
    assert t.equal(y[1], x), 'batch 1 does not match x'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_broadcast_batch(x: Tensor, b: int) -> Tensor:
    return repeat(x, 'c h w -> b c h w', b=b)
```
</details>

### Exercise 2 — per-token weight → per-feature weight

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply repeat to add a trailing feature axis that lets a per-token weight broadcast against a per-token-per-feature tensor.
> Keywords: match-shape, trailing-axis, attention-mask
> ```

**KCs targeted:** `repeat-match-broadcast-shape`

Implement `ex2_per_token_to_per_feature(w, d)` so each per-token weight is materialized across `d` feature columns.

Input shape: `(b, t)`. Output shape: `(b, t, d)`. Every `y[b, t, :]` should equal `w[b, t]`.

This is the shape you need when you want to scale a feature tensor of shape `(b, t, d)` by a per-token weight.

In [ ]:
def ex2_per_token_to_per_feature(w: Tensor, d: int) -> Tensor:
    """Repeat (b, t) → (b, t, d) by replicating w across the feature dim."""
    raise NotImplementedError()


def _test_ex2():
    w = t.tensor([[0.1, 0.5, 0.9, 0.0], [0.2, 0.3, 0.4, 0.5]])  # (b=2, t=4)
    y = ex2_per_token_to_per_feature(w, d=3)
    assert y.shape == (2, 4, 3), f'expected (2,4,3), got {y.shape}'
    # Every feature column should match the source per-token weight.
    for di in range(3):
        assert t.allclose(y[:, :, di], w), f'feature column {di} does not match w'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_per_token_to_per_feature(w: Tensor, d: int) -> Tensor:
    return repeat(w, 'b t -> b t d', d=d)
```
</details>

### Exercise 3 — vertical stretch (row-stretch via composition)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `(h r)` composition on the output side to stretch an axis — every source row appears `r` times consecutively (nearest-neighbor stretch).
> Keywords: stretch, composition, factor-order
> ```

**KCs targeted:** `repeat-stretch-via-composition`

Implement `ex3_stretch_vertical(x, r)` to vertically stretch a 2-D image: each row of the input should appear `r` times consecutively in the output.

Input shape: `(h, w)`. Output shape: `(h*r, w)`. Row pattern: `[x[0], x[0], ..., x[1], x[1], ..., x[2], ...]` with each input row appearing `r` times.

Compose `(h r)` on the **output** side — order matters. The inner factor `r` varies fastest, so positions `0..r-1` come from source row 0.

In [ ]:
def ex3_stretch_vertical(x: Tensor, r: int) -> Tensor:
    """Repeat (h, w) → (h*r, w), each source row appears r times in a block."""
    raise NotImplementedError()


def _test_ex3():
    x = t.arange(3 * 4).reshape(3, 4).float()
    y = ex3_stretch_vertical(x, r=2)
    assert y.shape == (6, 4), f'expected (6,4), got {y.shape}'
    # Source row i should appear at output rows i*r .. i*r+r-1.
    assert t.equal(y[0], x[0]) and t.equal(y[1], x[0]), 'block 0 does not match x[0]'
    assert t.equal(y[2], x[1]) and t.equal(y[3], x[1]), 'block 1 does not match x[1]'
    assert t.equal(y[4], x[2]) and t.equal(y[5], x[2]), 'block 2 does not match x[2]'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_stretch_vertical(x: Tensor, r: int) -> Tensor:
    return repeat(x, 'h w -> (h r) w', r=r)
```

**Why `(h r)` and not `(r h)`?** `(h r)` says the output's leading axis is composed of `h` outer blocks of size `r` each — so source row 0 fills positions `0..r-1`, source row 1 fills positions `r..2r-1`, etc. This is the **stretch** pattern (nearest-neighbor upsample). Swap the factor order to get **tiling** instead — see Exercise 4.
</details>

### Exercise 4 — horizontal tile (sequence-tile via composition)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `(r w)` composition on the output side to tile an axis — the full original sequence appears `r` times in a row.
> Keywords: tile, composition, factor-order
> ```

**KCs targeted:** `repeat-tile-via-composition`

Implement `ex4_tile_horizontal(x, r)` to horizontally tile a 2-D image: each row of the output is `r` concatenated copies of the same input row.

Input shape: `(h, w)`. Output shape: `(h, r*w)`. Each row of the output is `[x[i], x[i], ..., x[i]]` (r times concatenated).

Compose `(r w)` on the **output** side — the inner factor `w` varies fastest, so positions `0..w-1` of every tile match the original row.

In [ ]:
def ex4_tile_horizontal(x: Tensor, r: int) -> Tensor:
    """Repeat (h, w) → (h, r*w), each row is r copies of x[i] concatenated."""
    raise NotImplementedError()


def _test_ex4():
    x = t.arange(3 * 4).reshape(3, 4).float()
    y = ex4_tile_horizontal(x, r=2)
    assert y.shape == (3, 8), f'expected (3,8), got {y.shape}'
    # Each row of y should be [x[i], x[i]] concatenated.
    for i in range(3):
        assert t.equal(y[i, :4], x[i]), f'tile 0 of row {i} does not match x[{i}]'
        assert t.equal(y[i, 4:], x[i]), f'tile 1 of row {i} does not match x[{i}]'
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_tile_horizontal(x: Tensor, r: int) -> Tensor:
    return repeat(x, 'h w -> h (r w)', r=r)
```

**Stretch vs tile — factor order matters.**
- `(r w)` → outer factor is `r`, so the full `w`-long row appears `r` times consecutively → **tiling**.
- `(w r)` → outer factor is `w`, inner factor `r` → each source column appears `r` times before moving on → **stretching**.

Same factor names, opposite layouts. Always think about which factor varies fastest in the output's flat memory layout.
</details>

### Exercise 5 — 2×2 nearest-neighbor upsample (decompose + stretch + compose)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize new-axis introduction with axis composition to perform 2-D nearest-neighbor upsampling.
> Keywords: upsample, nearest-neighbor, integration, multi-kc
> ```

**KCs targeted:** `repeat-add-axis`, `repeat-stretch-via-composition`, `repeat-nearest-upsample`

Implement `ex5_upsample_2x2(x)` to nearest-neighbor upsample a batch of feature maps by 2× in both spatial dimensions.

Input shape: `(b, c, h, w)`. Output shape: `(b, c, 2h, 2w)`. Each input pixel `x[..., i, j]` should appear as a `2×2` block at output positions `y[..., 2i:2i+2, 2j:2j+2]`.

Introduce two new axes `p1, p2` of size 2, then compose them with the spatial axes so each pixel stretches into a 2×2 block.

Equivalent to `torch.nn.functional.interpolate(x, scale_factor=2, mode='nearest')`.

> ⚠️ **Integrative exercise.** This combines 3+ KCs in one pattern; empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_upsample_2x2(x: Tensor) -> Tensor:
    """Nearest-neighbor 2x2 upsample. (b, c, h, w) → (b, c, 2h, 2w).

    Each input pixel becomes a 2x2 block of identical values.
    """
    raise NotImplementedError()


def _test_ex5():
    x = t.arange(1 * 1 * 2 * 2).reshape(1, 1, 2, 2).float()
    y = ex5_upsample_2x2(x)
    assert y.shape == (1, 1, 4, 4), f'expected (1,1,4,4), got {y.shape}'
    expected = F.interpolate(x, scale_factor=2, mode='nearest')
    assert t.equal(y, expected), 'values differ from F.interpolate(scale=2, mode=nearest)'

    # Also test a larger random tensor.
    x2 = t.randn(2, 3, 5, 7)
    y2 = ex5_upsample_2x2(x2)
    assert y2.shape == (2, 3, 10, 14), f'expected (2,3,10,14), got {y2.shape}'
    assert t.allclose(y2, F.interpolate(x2, scale_factor=2, mode='nearest')), 'random-input mismatch'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_upsample_2x2(x: Tensor) -> Tensor:
    return repeat(
        x,
        'b c h w -> b c (h p1) (w p2)',
        p1=2, p2=2,
    )
```

**Reading the pattern.**
- `(h p1)` on the output side: source row `i` fills output rows `i*p1 .. i*p1+p1-1`. Same shape recipe as Exercise 3, but applied to an axis already present in the input.
- `(w p2)` does the same horizontally.
- `b` and `c` pass through untouched.

**Stretch vs tile here.** `(h p1)` stretches — every source pixel fills a contiguous 2×2 patch. If you wrote `(p1 h)` instead you'd get the entire row tiled twice vertically, which is **not** nearest-neighbor upsampling.
</details>

## Done

Run the cell below to report your progress to Delta Drills. The beacon fires only if all 5 exercises passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1', 'ex2', 'ex3', 'ex4', 'ex5'}

def _dd_feedback_level(num_passed: int) -> str:
    """Map exercise-pass count → arena-rating feedback enum."""
    if num_passed == 5: return 'not_much'   # 5/5 → felt easy
    if num_passed >= 3: return 'somewhat'
    return 'a_lot'

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {len(missing)} exercises still failing: {sorted(missing)}.")
        print("[Delta Drills] not reporting until all 5 pass.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}',
        'subtopics': [DD_SUBTOPIC],
        'feedback': _dd_feedback_level(len(_dd_passed)),
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()